# E3 재순위 — MedCPT cross-encoder (11/9, 구글 드라이브에 이어서 저장)

시스템 게이트+RRF 순위의 상위 N건을 MedCPT cross-encoder(`ncbi/MedCPT-Cross-Encoder`, NCBI 공개)로 다시 매겨 NFCorpus **dev**에서 잰다. API 비용 없음.

**작업 폴더가 구글 드라이브에 있다** (`MyDrive/KoMedRAG_E3_rerank`). 점수는 질의마다 드라이브의 캐시 파일에 확정되므로
**세션이 끊기면: 런타임 다시 연결 → 셀 1·2 → 끊긴 셀을 다시 실행.** 이미 매긴 쌍은 건너뛰고 이어서 간다. 결과도 드라이브에 남는다.

1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**
2. 셀 1: 드라이브 연결. **처음 한 번만** `mesh/bench/e3_rerank_bundle.zip` 업로드 창이 뜬다(드라이브에 저장되어 다음부터는 안 뜬다)
3. 셀 2 → 셀 3(상위 100) → 셀 4(상위 30) → 셀 5(결과 보기)

질의 변형: **raw** = 영어 원문(한국어 기사라면 LLM 번역이 있을 때) · **mesh** = MeSH 표목명 키워드(한국어 기사에서 LLM 없이 갈 때).
시험 분할(test)은 dev 결과로 변형과 N을 고정한 뒤 `--final`로 한 번만 돌린다.

In [ ]:
# 셀 1 — 드라이브 연결과 작업 폴더 (세션이 끊기면 여기부터 다시)
from google.colab import drive, files
import os, zipfile
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/KoMedRAG_E3_rerank'   # 바꿔도 된다
REFRESH = False      # 새 묶음 zip 으로 코드·데이터를 바꿀 때만 True (점수 캐시는 그대로 남는다)
os.makedirs(WORK, exist_ok=True)
BUNDLE = os.path.join(WORK, 'e3_rerank_bundle.zip')
if REFRESH or not os.path.isfile(os.path.join(WORK, 'bench', 'rerank_nfcorpus.py')):
    if REFRESH or not os.path.isfile(BUNDLE):
        print('e3_rerank_bundle.zip 을 올리세요 (드라이브에 저장됩니다)')
        up = files.upload()
        open(BUNDLE, 'wb').write(up[next(iter(up))])
    zipfile.ZipFile(BUNDLE).extractall(WORK)
os.chdir(os.path.join(WORK, 'bench'))
cache = 'rerank_cache_medcpt.jsonl'
n = sum(1 for _ in open(cache, encoding='utf-8')) if os.path.isfile(cache) else 0
print('작업 폴더:', os.getcwd(), '· 이미 매긴 쌍:', f'{n:,}')

In [ ]:
# 셀 2 — 라이브러리와 GPU 확인
!pip -q install transformers
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 — 런타임 유형을 T4 GPU 로 바꾸세요 (CPU 면 수십 분)')

In [ ]:
# 셀 3 — dev, 상위 100, 변형 둘(raw, mesh). 질의·문서 쌍 약 4만, T4 에서 몇 분. 끊기면 셀 1·2 뒤 이 셀만 다시
os.chdir(os.path.join(WORK, 'bench'))
!python rerank_nfcorpus.py --split dev --variants raw,mesh --top 100 --scorer medcpt

In [ ]:
# 셀 4 — dev, 상위 30 (시스템 기본값 RERANK_TOP=30). 셀 3 의 점수를 캐시에서 다시 쓰므로 거의 바로 끝난다
os.chdir(os.path.join(WORK, 'bench'))
!python rerank_nfcorpus.py --split dev --variants raw,mesh --top 30 --scorer medcpt

In [ ]:
# 셀 5 — 결과 보기 (파일은 드라이브의 작업 폴더 bench/ 와 bench/preds_v2/ 에 있다)
for n in (100, 30):
    p = os.path.join(WORK, 'bench', f'report_ir_nfcorpus_dev_rerank_medcpt_top{n}.md')
    if os.path.isfile(p): print(open(p, encoding='utf-8').read())

In [ ]:
# 셀 6 — (선택) 결과를 zip 으로 내려받기. 드라이브에 이미 있으므로 꼭 할 필요는 없다
import glob
os.chdir(os.path.join(WORK, 'bench'))
with zipfile.ZipFile('/content/e3_rerank_dev_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob('report_ir_*rerank*.md') + glob.glob('preds_v2/*medcpt*.trec') + glob.glob('rerank_cache_medcpt.jsonl'):
        z.write(p)
files.download('/content/e3_rerank_dev_results.zip')